# Load GIRAI 2026 → `fso_market_intelligence.frontier_labs`

**Global Index on Responsible AI (GIRAI), 2026 edition** — 135 countries scored 0–100 on responsible-AI *governance*. Complements the Oxford Government AI Readiness Index (capability) already in this schema.

Scoring hierarchy: overall `girai` score → **5 dimensions** (Inclusion & Diversity, Ethics & Sustainability, Labour & Skills, Trust & Safety, AI Use in Public Service) × **3 pillars** (AI Policy, CSO Engagement, Enabling Conditions) → **38 thematic indicators**. `urai_penalty` docks countries with unacceptable-risk AI systems.

Builds 5 tidy Delta tables (overwrite each run):

| Table | Grain | ~Rows |
|---|---|---|
| `girai_scores` | 1 row/country: rank, girai, urai_penalty, 5 dims, 3 pillars | 135 |
| `girai_indicators` | country × indicator (long) | 5,130 |
| `girai_pillar_dimension_scores` | country × dimension × pillar (long) | 2,025 |
| `girai_editions_comparison` | country × indicator, 2024 vs 2026 | 1,820 |
| `girai_country_classifications` | country → region / income / GDP / LDC | 138 |

Source: 3 xlsx in the `GIRAI/` Git folder. ISO3 codes ship with the data (no country fuzzy-matching needed).

In [ ]:
%pip install openpyxl -q

In [ ]:
CATALOG, SCHEMA, EDITION = "fso_market_intelligence", "frontier_labs", 2026

import subprocess, os, re

def _find(fname):
    hits = subprocess.run(["find", "/Workspace", "-maxdepth", "9", "-name", fname],
                          capture_output=True, text=True).stdout.strip().splitlines()
    assert hits, f"{fname} not found under /Workspace — pull the Git folder first"
    return hits[0]

XLSX_SCORES = _find("GIRAI_rankings_and_scores.xlsx")
XLSX_ED     = _find("GIRAI_editions_comparison.xlsx")
XLSX_REG    = _find("GIRAI_regions_and_subregions.xlsx")
print("scores  =", XLSX_SCORES)
print("editions=", XLSX_ED)
print("regions =", XLSX_REG)

# snake_case a messy Excel header into a safe Delta column name
san = lambda c: re.sub(r"[^0-9a-zA-Z]+", "_", str(c)).strip("_").lower()

In [ ]:
import pandas as pd
from pyspark.sql import functions as F

def write_table(pdf, name):
    sdf = (spark.createDataFrame(pdf)
           .withColumn("edition", F.lit(EDITION))
           .withColumn("captured_at", F.current_date()))
    (sdf.write.mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{name}"))
    print(f"  {name:32} {sdf.count():>5} rows")
    return sdf

### Tables 1–3 — from `GIRAI_rankings_and_scores.xlsx`

`girai_scores` has a two-row header (row 0 = column names + merged `DIMENSIONS SCORES` / `PILLAR SCORES` group markers, row 1 = the dimension/pillar labels), so we skip both rows and assign explicit clean names. Indicators and pillar×dimension scores are melted to long format — far easier for Genie to query than 38 / 15 wide columns with spaces in the names.

In [ ]:
# ── 1. girai_scores (1 row/country) ──
SCORE_COLS = ["ranking", "iso3", "country", "region", "girai", "girai_raw", "urai_penalty",
              "dim_inclusion_diversity", "dim_ethics_sustainability", "dim_labour_skills",
              "dim_trust_safety", "dim_ai_public_service",
              "pillar_ai_policy", "pillar_cso_engagement", "pillar_enabling_conditions"]
scores = pd.read_excel(XLSX_SCORES, "ranking_and_scores", header=None, skiprows=2, names=SCORE_COLS)
scores = scores[scores["iso3"].notna()].copy()
scores["ranking"] = scores["ranking"].astype(int)

# ── 2. girai_indicators (long: country × indicator) ──
ind = pd.read_excel(XLSX_SCORES, "all_indicators", header=0)
ind = ind[ind["iso3"].notna()].copy()
ind_vals = [c for c in ind.columns if c not in ("iso3", "country", "girai", "girai_raw")]
indicators = ind.melt(id_vars=["iso3", "country"], value_vars=ind_vals,
                      var_name="indicator", value_name="score")

# ── 3. girai_pillar_dimension_scores (long: country × dimension × pillar) ──
pdm = pd.read_excel(XLSX_SCORES, "pillar_scores_by_dimension", header=0)
pdm = pdm[pdm["iso3"].notna()].copy()
pdm_vals = [c for c in pdm.columns if c not in ("ranking", "iso3", "country")]
pdmL = pdm.melt(id_vars=["iso3", "country"], value_vars=pdm_vals,
                var_name="_raw", value_name="score")
pdmL["_clean"] = pdmL["_raw"].str.replace(r"\s*\(0-100\)\s*$", "", regex=True)
# split on the FIRST ' in ' only — the dimension 'AI Use in Public Service' contains ' in '
pdmL[["pillar", "dimension"]] = pdmL["_clean"].str.split(" in ", n=1, expand=True)
pillar_dimension = pdmL.drop(columns=["_raw", "_clean"])[["iso3", "country", "pillar", "dimension", "score"]]

print("Writing score tables:")
write_table(scores, "girai_scores")
write_table(indicators, "girai_indicators")
write_table(pillar_dimension, "girai_pillar_dimension_scores")

### Table 4 — editions comparison (2024 vs 2026)

From `GIRAI_editions_comparison.xlsx` (`DB` sheet): per country × indicator, the framework status / title / link and initiative & CSO-engagement existence flags in both editions — the "what changed" surface.

In [ ]:
ed = pd.read_excel(XLSX_ED, "DB", header=0)
ed = ed[ed["ISO3"].notna()].copy()
ed.columns = [san(c) for c in ed.columns]
print("Writing editions table:")
write_table(ed, "girai_editions_comparison")

### Table 5 — country classifications

From `GIRAI_regions_and_subregions.xlsx` (`girai_regions_2026` sheet): region, subregion, developing status, least-developed-country flag, World Bank income group, and GDP per capita (PPP). Join dimension for slicing scores by geography / income. Has 138 rows (135 scored countries + 3 extra reference territories); joins on `iso3` so the extras never surface.

In [ ]:
cc = pd.read_excel(XLSX_REG, "girai_regions_2026", header=0)
cc = cc[cc["ISO3"].notna()].copy()
cc.columns = [san(c) for c in cc.columns]
print("Writing classifications table:")
write_table(cc, "girai_country_classifications")

### Sanity check

In [ ]:
print("Top 10 countries by GIRAI 2026 score:")
display(spark.table(f"{CATALOG}.{SCHEMA}.girai_scores").orderBy("ranking").limit(10))